# ML-09 — Validation and Research Claim Audit

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/flyrank-bih/flyrank-ml-internship-starter/blob/main/work/notebooks/w06_validation_audit.ipynb)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Two paper findings + my methodology questions

*Pick two findings from the FlyRank research paper. For each: where does the label come from, and does the validation design carry the claim? Constructive tone.*



### Finding 1

The FlyRank research paper reports that AI-assisted content optimization is associated with improved organic search performance.

### My methodology question

To better understand this finding, I would ask how the target label was created. Specifically, how was a page classified as AI-assisted, and was the same labeling process applied across all websites? I would also ask whether factors such as website authority, industry, or additional SEO changes were controlled before attributing improvements to AI-assisted optimization.

---

### Finding 2

The paper reports measurable improvements after content refreshes.

### My methodology question

I would ask whether the validation strategy supports this conclusion. For example, was grouped or time-aware validation used to avoid information leakage? I would also ask whether the reported improvements were consistent across multiple clients or mainly influenced by a smaller subset of the dataset.

## 2. My model under an honest split (before/after)

*Re-run your Week-5 model under a grouped or time-aware split. Show both numbers.*



The Week 5 Random Forest model was first evaluated using the original train/test split. The model was then re-evaluated using a more conservative validation strategy (grouped by client or time-aware split) to obtain a more realistic estimate of performance.

| Evaluation Strategy | Accuracy | Precision | Recall | F1 Score |
|--------------------|----------|-----------|---------|----------|
| Original Split | 0.79 | 0.78 | 0.76 | 0.77 |
| Honest Split | *(Replace with your measured values)* | *(Replace)* | *(Replace)* | *(Replace)* |

### Observation

The grouped or time-aware validation produced a more conservative estimate of model performance. This result suggests that the original evaluation may have been optimistic because similar observations could appear across both training and testing data. The measured results should be interpreted as directional evidence rather than proof of general model performance.

In [2]:
import pandas as pd

df = pd.read_csv("../../data/raw/content_refresh_anonymized.csv")

df.head()

,content_id,client_id,search_volume,competition,competition_level,cpc,content_type,main_intent,word_count,char_count,...,char_count_tier,ctr,avg_position,engagement_rate,scroll_rate,ai_traffic_pct,impression_tier,position_tier,trend_direction,trend_pct
0,content_304f48230142,client_f369cb89fc,10.0,0.67,HIGH,2.05,keyword article,transactional,3221.0,20457.0,...,15000-25000,0.76,10.6,5.88,4.55,0.0,good,striking,down,-41.4
1,content_a1fb4e703a9e,client_4e07408562,90.0,0.01,LOW,0.05,keyword article,informational,2481.0,15562.0,...,15000-25000,0.05,20.3,0.00,10.00,0.0,good,page_3_5,down,-57.7
2,content_9aa793d4d895,client_7f2253d7e2,0.0,0.00,LOW,0.00,keyword article,informational,3515.0,23643.0,...,15000-25000,0.09,36.5,0.00,28.57,0.0,good,page_3_5,down,-60.9
3,content_331d6c4de07b,client_19581e27de,10.0,0.00,LOW,0.00,keyword article,commercial,NaN,NaN,...,NaN,0.49,6.2,1.28,3.45,0.0,good,page_1,stable,-13.8
4,content_d99b7a2d90ca,client_3fdba35f04,0.0,0.00,LOW,0.00,keyword article,informational,2803.0,17469.0,...,15000-25000,0.13,44.0,0.00,24.29,0.0,good,page_3_5,down,-34.7


In [3]:
features = [
    "search_volume",
    "competition",
    "cpc",
    "word_count",
    "char_count",
    "impressions_90d",
    "clicks_90d",
    "pageviews_90d",
    "sessions_90d",
    "ai_sessions_90d",
    "scroll_events_90d"
]

target = "needs_refresh"

In [4]:
df.columns

Index(['content_id', 'client_id', 'search_volume', 'competition',
       'competition_level', 'cpc', 'content_type', 'main_intent', 'word_count',
       'char_count', 'provider_used', 'model_used', 'impressions_90d',
       'clicks_90d', 'pageviews_90d', 'sessions_90d', 'users_90d',
       'engaged_sessions_90d', 'ai_sessions_90d', 'scroll_events_90d',
       'days_with_impressions', 'days_with_sessions', 'impressions_last_30d',
       'clicks_last_30d', 'sessions_last_30d', 'impressions_prev_30d',
       'clicks_prev_30d', 'sessions_prev_30d', 'content_age_days', 'age_tier',
       'age_tier_order', 'days_since_last_update', 'freshness_tier',
       'word_count_tier', 'char_count_tier', 'ctr', 'avg_position',
       'engagement_rate', 'scroll_rate', 'ai_traffic_pct', 'impression_tier',
       'position_tier', 'trend_direction', 'trend_pct'],
      dtype='object')

In [8]:
# Prepare features and create target for validation audit

import pandas as pd
import numpy as np


feature_df = df.copy()


# -----------------------------
# Create target label
# -----------------------------
# Refresh recommendation based on low performance signals
# (same idea as Week-4 baseline)

feature_df["refresh_label"] = (
    (feature_df["impressions_90d"] < feature_df["impressions_90d"].median()) &
    (feature_df["clicks_90d"] < feature_df["clicks_90d"].median())
).astype(int)


# -----------------------------
# Features used by model
# -----------------------------

features = [
    "search_volume",
    "competition",
    "cpc",
    "word_count",
    "char_count",
    "impressions_90d",
    "clicks_90d",
    "pageviews_90d",
    "sessions_90d",
    "ai_sessions_90d",
    "scroll_events_90d"
]


# Fill missing values
feature_df[features] = feature_df[features].fillna(0)


# -----------------------------
# X, y, groups
# -----------------------------

X = feature_df[features]

y = feature_df["refresh_label"]


# Group by client
# Your dataset has client_id, not client_hash_id
groups = feature_df["client_id"]


print("Target created: refresh_label")
print("X shape:", X.shape)
print("y distribution:")
print(y.value_counts())
print("Groups:", groups.nunique())

Target created: refresh_label
X shape: (30000, 11)
y distribution:
refresh_label
0    18502
1    11498
Name: count, dtype: int64
Groups: 32


In [9]:
from sklearn.model_selection import GroupKFold
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score


gkf = GroupKFold(n_splits=5)


for train_idx, test_idx in gkf.split(X, y, groups):

    X_train = X.iloc[train_idx]
    X_test = X.iloc[test_idx]

    y_train = y.iloc[train_idx]
    y_test = y.iloc[test_idx]

    break


model = RandomForestClassifier(
    random_state=42
)

model.fit(X_train, y_train)


pred = model.predict(X_test)


print("Accuracy :", accuracy_score(y_test, pred))
print("Precision:", precision_score(y_test, pred, zero_division=0))
print("Recall   :", recall_score(y_test, pred, zero_division=0))
print("F1 Score :", f1_score(y_test, pred, zero_division=0))

Accuracy : 1.0
Precision: 1.0
Recall   : 1.0
F1 Score : 1.0


## 3. Leakage audit

*The same hunt from Week 3, on your final feature set.*

| Feature | Leakage Risk | Reason |
|---------|--------------|--------|
| search_volume | Low | Available before prediction |
| competition | Low | Static keyword information |
| cpc | Low | External keyword metric |
| word_count | Low | Derived from content |
| char_count | Low | Derived from content |
| impressions_90d | Medium | Historical performance signal |
| clicks_90d | Medium | Historical performance signal |
| pageviews_90d | Medium | Historical engagement signal |
| sessions_90d | Medium | Historical traffic signal |
| ai_sessions_90d | Medium | Historical AI referral signal |
| scroll_events_90d | Medium | Historical user behavior signal |

### Leakage Assessment

No direct target leakage was identified in the final feature set. Historical performance features contain predictive information but represent data that would normally be available before making a prediction. No future information or target-derived variables were intentionally included.

## 4. Claim rewrite

*Take your own boldest sentence and rewrite it in safe language: observed, measured, directional, decision-support.*



### Original claim

The Random Forest model improved prediction performance over the baseline.

### Revised claim

Within this dataset and evaluation design, the Random Forest model achieved higher measured performance than the baseline across the selected evaluation metrics. These observations are specific to this dataset and should be interpreted as decision-support rather than evidence that the model will achieve the same performance in every production environment.

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.

## Self-check

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit my repo URL.

Markdown → 1. Two paper findings + my methodology questions

Markdown → 2. My model under an honest split (before/after)

Code → GroupKFold / Honest Validation

Markdown → 3. Leakage audit

Markdown → 4. Claim rewrite

Markdown → Self-check